# Chapter 16 &mdash; The Cook&ndash;Levin Theorem: 3-SAT is NP-Complete

**Concept 8 of the Chapter 16 decomposition:** *The Cook–Levin Theorem: 3-SAT is NP-Complete*

Encode an NDTM's computation tableau as a 3-CNF that is satisfiable iff the machine accepts.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16/Concept-Cook-Levin/Concept-Cook-Levin.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


> **Cook&ndash;Levin.** SAT is NP-complete, and so is 3-SAT.

The proof encodes an NDTM's computation as a **tableau**: a $p(n)\times p(n)$ grid
whose row $i$ is the configuration at step $i$. A variable $x_{i,j,s}$ says "cell
$(i,j)$ holds symbol $s$".

Four families of clauses:

* **cell** &mdash; each cell holds exactly one symbol;
* **start** &mdash; row 0 is the initial configuration;
* **accept** &mdash; some cell holds the accept state;
* **move** &mdash; every $2\times3$ **window** is legal for $\delta$.

The move clauses are the heart, and they work for the same reason the PCP tiles did
(Chapter 15): a TM step is **local**. The formula has $O(p(n)^2)$ variables, so the
construction is polynomial &mdash; and it is satisfiable exactly when an accepting
computation exists.

## 2. Definitions

### Tableau variables and their count

In [ ]:
# --- a tiny CNF toolkit -------------------------------------------------
# A literal is an int: 3 means x3, -3 means NOT x3.
# A clause is a tuple of literals; a formula is a list of clauses.
from itertools import product

def nvars(F):
    return max((abs(l) for c in F for l in c), default=0)

def evaluate(F, assign):
    # assign: dict var -> bool
    return all(any(assign[abs(l)] == (l > 0) for l in c) for c in F)

def brute_sat(F):
    n = nvars(F)
    for bits in product([False, True], repeat=n):
        a = {i + 1: bits[i] for i in range(n)}
        if evaluate(F, a): return a
    return None

def show_cnf(F):
    def lit(l): return ("x%d" % l) if l > 0 else ("~x%d" % -l)
    return " AND ".join("(" + " OR ".join(lit(l) for l in c) + ")" for c in F)


def tableau_vars(p, symbols):
    # x[i][j][s] : cell (i,j) holds symbol s
    idx, n = {}, 0
    for i in range(p):
        for j in range(p):
            for s in symbols:
                n += 1; idx[(i, j, s)] = n
    return idx, n

def cell_clauses(idx, p, symbols):
    F = []
    for i in range(p):
        for j in range(p):
            F.append(tuple(idx[(i, j, s)] for s in symbols))          # at least one
            for a in range(len(symbols)):
                for b in range(a + 1, len(symbols)):                  # at most one
                    F.append((-idx[(i, j, symbols[a])],
                              -idx[(i, j, symbols[b])]))
    return F

### The 2x3 window legality test

In [ ]:
def windows(p):
    return [(i, j) for i in range(p - 1) for j in range(p - 2)]

def legal_windows(delta, symbols, states):
    # enumerate the 2x3 windows consistent with delta -- finitely many
    ok = set()
    for top in product(symbols, repeat=3):
        for bot in product(symbols, repeat=3):
            if not any(s in states for s in top):
                if top == bot: ok.add((top, bot))          # nothing happens here
            else:
                ok.add((top, bot))                          # delta decides; abbreviated
    return ok

## 3. Tests

The tableau's size is polynomial in the input.

In [ ]:
symbols = ['0', '1', '.', 'q0', 'qa']
for n in [2, 4, 8, 16]:
    p = n * n                      # a polynomial time bound
    idx, nv = tableau_vars(min(p, 12), symbols)
    print("  n=%2d : p(n)=%3d, tableau %dx%d, variables %s"
          % (n, p, min(p, 12), min(p, 12), format(nv, ',')))
print("\nO(p(n)^2 * |Gamma|) variables -- polynomial, which is the point.")

**Cell clauses:** every cell holds exactly one symbol.

In [ ]:
idx, nv = tableau_vars(2, ['0', '1'])
F = cell_clauses(idx, 2, ['0', '1'])
print("2x2 tableau over {0,1} : %d variables, %d cell clauses" % (nv, len(F)))
print(show_cnf(F[:4]))
a = brute_sat(F)
print("\nsatisfiable? ", a is not None)
assert a is not None

Adding a contradiction makes it unsatisfiable, as it should.

In [ ]:
G = F + [(idx[(0, 0, '0')],), (idx[(0, 0, '1')],)]   # cell 0,0 is both
print("forcing cell (0,0) to hold both symbols :", brute_sat(G) is None)
assert brute_sat(G) is None

**Locality:** only a $2\times3$ window can change, so finitely many are legal.

In [ ]:
p = 4
print("windows in a %dx%d tableau : %d" % (p, p, len(windows(p))))
sym3 = ['0', '1', 'q']
ok = legal_windows(None, sym3, {'q'})
print("possible windows over %d symbols : %d" % (len(sym3), len(sym3) ** 6))
print("legal ones (abbreviated rule)    : %d" % len(ok))
assert len(ok) < len(sym3) ** 6
print("\nThe legal set is computed ONCE from delta, then reused at every window.")

So the whole formula is polynomial, and satisfiable iff the NDTM accepts.

In [ ]:
print("clauses: cell O(p^2)  start O(p)  accept O(p^2)  move O(p^2)")
print("total  : O(p(n)^2) clauses over O(p(n)^2) variables")
print()
print("An assignment IS a tableau; a legal tableau IS an accepting computation.")
print("So: satisfiable  <=>  the NDTM accepts.  That is Cook-Levin.")

And 3-SAT follows by the clause-splitting of Concept 7.

In [ ]:
big = [(1, 2, 3, 4, 5)]
def split(c_, nxt):
    out = []
    c_ = list(c_)
    while len(c_) > 3:
        nxt += 1
        out.append((c_[0], c_[1], nxt))
        c_ = [-nxt] + c_[2:]
    out.append(tuple(c_))
    return out
print("one 5-literal clause ->", show_cnf(split(big[0], 5)))
print("\nEquisatisfiable, and every clause now has exactly three literals.")

## 4. Exercises


1. Write the start clauses for input `0110` and a 6-wide tableau.
2. Why is a $2\times3$ window enough? Would $2\times2$ do?
3. Count the clauses precisely for $p(n)=n^2$ and $|\Gamma|=5$.

In [ ]:
# Your work for the exercises above.